# `colab_qwen25_3b_variant_A.ipynb` — feature-ablation Variant A

**Variant A:** `canonical embedding only`
**Feature keys added on top of canonical:** `(none — canonical only)`
**Model tag (must match the parent run):** `colab_qwen25_3b`

This notebook is a **CPU-only** classifier-ablation experiment. It does NOT load the LLM, does NOT regenerate data, and does NOT touch the 7 downstream datasets. It just reads the pre-computed `colab_qwen25_3b_dataset_full.json` produced by the main notebook's BLOCK 6.5, trains the 5-layer MLP on the variant-specific feature subset, and reports Wikipedia held-out metrics.

**Inputs (must exist in the same directory):**
* `colab_qwen25_3b_dataset_full.json` — every generated record, both classes, pre-split. Produced by the main `project_colab_qwen25_3b.ipynb` (BLOCK 6.5).

**Outputs (written to the same directory):**
* `colab_qwen25_3b_variant_A_results.json` — full audit (env, config, label distribution, raw `{y,p,prob}` for first 50 test samples, all metrics, MLP epoch history).
* `colab_qwen25_3b_variant_A_best.pth` — best MLP weights + scaler tensors.

**Variant menu (for cross-reference):**
| Variant | Features | Purpose |
|---|---|---|
| A | canonical embedding only | Baseline (mid-eval replicate target ≈ 0.673 AUROC on Qwen-3B) |
| B | canonical + D_mean | Drift contribution alone |
| C | canonical + V_last | Variance contribution alone |
| D | canonical + H_mean | Entropy contribution alone |
| E | canonical + D_mean + V_last + H_mean | Headline (matches main notebook's BLOCK 9) |

Acceptance criterion for the headline ablation: Variant **E** AUROC > Variant **A** AUROC by at least **0.02** on the Wikipedia held-out test set.


In [ ]:
# =============================================================================
# BLOCK 1: IMPORTS + LOAD <MODEL_TAG>_dataset_full.json
# =============================================================================
import os, sys, json, random, math, time, datetime, platform
import numpy as np

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
except ImportError as e:
    raise SystemExit(f"torch is required. Install with `pip install torch`. err={e}")

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              roc_auc_score, confusion_matrix, brier_score_loss)

# -----------------------------------------------------------------------------
# Configuration — must match the parent run for the ablation to be comparable.
# -----------------------------------------------------------------------------
MODEL_TAG = "colab_qwen25_3b"
VARIANT   = "A"
SCALAR_KEYS = []   # subset of ["D_mean", "V_last", "H_mean"]
SEED      = 42
EPOCHS    = 10
BATCH     = 32
LR        = 5e-4
WD        = 1e-5
TEST_FRAC = 0.20

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DATASET_FULL_PATH = f"{MODEL_TAG}_dataset_full.json"
if not os.path.exists(DATASET_FULL_PATH):
    raise SystemExit(
        f"\n[ERROR] Required input file not found: {DATASET_FULL_PATH}\n"
        f"Run the main notebook (project_{MODEL_TAG}.ipynb) first — it writes\n"
        f"this file in BLOCK 6.5. Then re-run this variant notebook.\n"
    )

# -----------------------------------------------------------------------------
# Diagnostics dict — single-file audit, written at end of notebook
# -----------------------------------------------------------------------------
DIAG = {
    "schema_version": "ablation-1.0",
    "variant":        VARIANT,
    "variant_label":  'canonical embedding only',
    "model_tag":      MODEL_TAG,
    "scalar_keys":    SCALAR_KEYS,
    "config": {
        "seed": SEED, "epochs": EPOCHS, "batch_size": BATCH,
        "lr": LR, "weight_decay": WD, "test_frac": TEST_FRAC,
    },
    "env": {
        "python":   sys.version.split()[0],
        "platform": platform.platform(),
        "torch":    torch.__version__,
        "numpy":    np.__version__,
    },
    "stage_timings_sec": {},
    "errors":   [],
}

# Load
_t0 = time.perf_counter()
with open(DATASET_FULL_PATH, "r") as f:
    records = json.load(f)
DIAG["stage_timings_sec"]["load_dataset"] = round(time.perf_counter() - _t0, 2)
DIAG["dataset_full_size_mb"] = round(os.path.getsize(DATASET_FULL_PATH) / 1e6, 2)
DIAG["n_records_total"] = len(records)

# Basic sanity
hidden_dim = len(records[0]["embedding"])
labels = [r["label"] for r in records]
DIAG["hidden_dim"] = int(hidden_dim)
DIAG["label_distribution_total"] = {
    "y=0": int(sum(1 for y in labels if y == 0)),
    "y=1": int(sum(1 for y in labels if y == 1)),
}
print(f"✓ Loaded {len(records)} records  (hidden_dim={hidden_dim})  "
      f"size={DIAG['dataset_full_size_mb']} MB  load_time={DIAG['stage_timings_sec']['load_dataset']}s")
print(f"  Label dist: y=0 -> {DIAG['label_distribution_total']['y=0']}   "
      f"y=1 -> {DIAG['label_distribution_total']['y=1']}")
print(f"  Variant {VARIANT}: feature keys to add on top of canonical = {SCALAR_KEYS}")


In [ ]:
# =============================================================================
# BLOCK 2: BUILD FEATURE MATRIX FOR VARIANT A + 80/20 SPLIT (seed=42)
# =============================================================================
# Shuffle deterministically with the same seed as the main notebook so the
# train/test split is identical across all 5 variants.
random.seed(SEED)
records_shuffled = list(records)
random.shuffle(records_shuffled)

split = int((1.0 - TEST_FRAC) * len(records_shuffled))
train_records = records_shuffled[:split]
test_records  = records_shuffled[split:]
DIAG["n_train"] = len(train_records)
DIAG["n_test"]  = len(test_records)


def build_X_y(recs, scaler=None, fit=False):
    canon = np.array([r["embedding"] for r in recs], dtype=np.float32)
    if SCALAR_KEYS:
        scalars = np.array([[r[k] for k in SCALAR_KEYS] for r in recs],
                            dtype=np.float32)
        if fit:
            scaler = StandardScaler().fit(scalars)
        scalars_z = scaler.transform(scalars).astype(np.float32)
        X = np.concatenate([canon, scalars_z], axis=1)
    else:
        # Variant A — canonical only, no scaler needed
        X = canon
    y = np.array([r["label"] for r in recs], dtype=np.int64)
    return X, y, scaler


X_tr, y_tr, scaler = build_X_y(train_records, fit=True)
X_te, y_te, _      = build_X_y(test_records,  scaler=scaler, fit=False)
input_dim = X_tr.shape[1]
DIAG["input_dim"] = int(input_dim)
print(f"✓ Feature matrix built. input_dim = {input_dim}")
print(f"  (canonical {hidden_dim}  +  scalars {len(SCALAR_KEYS)})")
print(f"  train n = {len(train_records)}   test n = {len(test_records)}")


In [ ]:
# =============================================================================
# BLOCK 3: 5-LAYER MLP + TRAIN  (same arch as main notebook BLOCK 8)
# =============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DIAG["device"] = str(device)


class MINDPlusClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64),  nn.ReLU(),
            nn.Linear(64, 2),
        )
    def forward(self, x):
        return self.layers(x)


class _FeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X); self.y = torch.from_numpy(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


mlp = MINDPlusClassifier(input_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(mlp.parameters(), lr=LR, weight_decay=WD)
train_loader = DataLoader(_FeatureDataset(X_tr, y_tr), batch_size=BATCH, shuffle=True)
test_loader  = DataLoader(_FeatureDataset(X_te, y_te), batch_size=BATCH, shuffle=False)

best_acc = 0.0
epoch_history = []
_t0 = time.perf_counter()
for ep in range(EPOCHS):
    mlp.train()
    ep_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = mlp(x)
        l = loss_fn(logits, y)
        l.backward()
        opt.step()
        ep_loss += float(l.item())
    # Held-out test acc per epoch
    mlp.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct += (mlp(x).argmax(1) == y).sum().item()
            total += y.size(0)
    acc_ep = correct / max(total, 1)
    epoch_history.append({"epoch": ep + 1,
                          "train_loss": round(ep_loss / max(len(train_loader), 1), 4),
                          "test_acc":   round(acc_ep, 4)})
    star = " ★" if acc_ep > best_acc else ""
    if acc_ep > best_acc:
        best_acc = acc_ep
        ckpt = {"model_state": mlp.state_dict(),
                "input_dim": input_dim,
                "variant": VARIANT,
                "scalar_keys": SCALAR_KEYS}
        if SCALAR_KEYS:
            ckpt["scaler_mean"]  = scaler.mean_
            ckpt["scaler_scale"] = scaler.scale_
        torch.save(ckpt, f"{MODEL_TAG}_variant_{VARIANT}_best.pth")
    print(f"epoch {ep+1:2d}/{EPOCHS}  train_loss={epoch_history[-1]['train_loss']:.4f}  test_acc={acc_ep:.4f}{star}")

DIAG["stage_timings_sec"]["mlp_train"] = round(time.perf_counter() - _t0, 2)
DIAG["mlp_epoch_history"] = epoch_history
DIAG["mlp_best_acc"] = float(best_acc)
print(f"✓ best test acc: {best_acc:.4f}   train time: {DIAG['stage_timings_sec']['mlp_train']}s")


In [ ]:
# =============================================================================
# BLOCK 4: WIKIPEDIA HELD-OUT EVALUATION
# =============================================================================
# Reload best checkpoint
_ckpt = torch.load(f"{MODEL_TAG}_variant_{VARIANT}_best.pth",
                    weights_only=False, map_location=device)
mlp.load_state_dict(_ckpt["model_state"])
mlp.eval()

all_p, all_y, all_pr = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = mlp(x)
        prob = torch.softmax(logits, dim=1)[:, 1]
        all_p.extend(logits.argmax(1).cpu().numpy())
        all_y.extend(y.numpy())
        all_pr.extend(prob.cpu().numpy())
all_p = np.array(all_p); all_y = np.array(all_y); all_pr = np.array(all_pr)


def _safe(fn, *a, **kw):
    try:    return float(fn(*a, **kw))
    except Exception as e:
        DIAG["errors"].append(f"metric {fn.__name__}: {e}")
        return float("nan")


acc  = _safe(accuracy_score, all_y, all_p)
prec, rec, f1, _ = precision_recall_fscore_support(all_y, all_p, average="binary",
                                                     zero_division=0)
auc   = _safe(roc_auc_score, all_y, all_pr) if len(set(all_y.tolist())) > 1 else float("nan")
brier = _safe(brier_score_loss, all_y, all_pr)
cm    = confusion_matrix(all_y, all_p, labels=[0, 1])

metrics = {
    "n": int(len(all_y)),
    "accuracy":  float(acc),
    "precision": float(prec),
    "recall":    float(rec),
    "f1":        float(f1),
    "auc_roc":   float(auc),
    "brier":     float(brier),
    "cm": {"tn": int(cm[0, 0]), "fp": int(cm[0, 1]),
           "fn": int(cm[1, 0]), "tp": int(cm[1, 1])},
    "label_distribution_test": {"y=0": int((all_y == 0).sum()),
                                "y=1": int((all_y == 1).sum())},
    "pred_distribution_test":  {"p=0": int((all_p == 0).sum()),
                                "p=1": int((all_p == 1).sum())},
    "prob_min":  round(float(all_pr.min()) if len(all_pr) else 0.0, 4),
    "prob_max":  round(float(all_pr.max()) if len(all_pr) else 0.0, 4),
    "prob_mean": round(float(all_pr.mean()) if len(all_pr) else 0.0, 4),
}
DIAG["wikipedia_eval"] = metrics

# Raw arrays for the first 50 test samples — for offline audit
DIAG["raw_y_pred_prob_first_50"] = [
    {"y": int(yy), "p": int(pp), "prob": round(float(prob), 4)}
    for yy, pp, prob in zip(all_y[:50], all_p[:50], all_pr[:50])
]

print(f"\nWikipedia held-out  (variant {VARIANT}, n={metrics['n']})")
print(f"  Accuracy : {metrics['accuracy']:.4f}")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall   : {metrics['recall']:.4f}")
print(f"  F1       : {metrics['f1']:.4f}")
print(f"  AUROC    : {metrics['auc_roc']:.4f}")
print(f"  Brier    : {metrics['brier']:.4f}")


In [ ]:
# =============================================================================
# BLOCK 5: DUMP RESULTS JSON
# =============================================================================
DIAG["timestamp_utc"] = (datetime.datetime.now(datetime.timezone.utc)
                          .replace(microsecond=0).isoformat().replace("+00:00", "Z"))
DIAG["stage_timings_sec"]["total"] = round(sum(DIAG["stage_timings_sec"].values()), 2)

out_path = f"{MODEL_TAG}_variant_{VARIANT}_results.json"
with open(out_path, "w") as f:
    json.dump(DIAG, f, indent=2, default=str)
print(f"\n✓ Wrote {out_path}  ({os.path.getsize(out_path)/1024:.1f} KB)")

# Single-row banner
print("\n" + "=" * 80)
print(f"VARIANT {VARIANT}  ({DIAG['variant_label']})")
print("=" * 80)
m = DIAG["wikipedia_eval"]
print(f"  AUROC    {m['auc_roc']:.4f}     Acc {m['accuracy']:.4f}     "
      f"F1 {m['f1']:.4f}     Brier {m['brier']:.4f}")
print(f"  input_dim {DIAG['input_dim']}     "
      f"train {DIAG['n_train']}     test {DIAG['n_test']}     "
      f"best_test_acc {DIAG['mlp_best_acc']:.4f}")
print("=" * 80)
print(f"\nPaste {out_path} back to the assistant for cross-variant comparison.")
